# 08 — Time Series: ARIMA, VAR y Causalidad de Granger
**Autores clave:** Box & Jenkins (1970) · Sims (1980) · Granger (1969) · Hamilton (1994) *Time Series Analysis* · Lütkepohl (2005) *New Introduction to Multiple Time Series*

## ARIMA(p,d,q)
$$\phi(L)(1-L)^d y_t = \theta(L)\varepsilon_t$$
- $p$: orden AR (Autoregressive)
- $d$: orden de integración (diferenciación)
- $q$: orden MA (Moving Average)

## VAR(p) — Vector Autoregression (Sims, 1980)
$$Y_t = A_1 Y_{t-1} + A_2 Y_{t-2} + \cdots + A_p Y_{t-p} + u_t$$
donde $Y_t$ es un vector de $K$ variables. Cada variable depende de sus propios lags y los lags de las demás.

> **Sims (1980):** Los modelos macro estructurales imponían restricciones arbitrarias. El VAR trata todas las variables simétricamente — dejar que los datos hablen.

## Causalidad de Granger (1969)
$x$ **Granger-causa** $y$ si los lags de $x$ ayudan a predecir $y$ más allá de los lags de $y$ solos:
$$H_0: \text{los coeficientes de } x_{t-k} \text{ en la ecuación de } y_t \text{ son todos cero}$$

> **Importante:** Causalidad de Granger es **predictiva**, no causal en sentido Pearl/Rubin.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
T = 400

# VAR(1) bivariado: PIB y desempleo (tipo curva de Phillips / Okun)
# y_t = A * y_{t-1} + u_t
# y[0] = log(PIB growth), y[1] = desempleo
A = np.array([[0.70, -0.30],   # PIB_t = 0.7 PIB_{t-1} - 0.3 desemp_{t-1}
              [-0.15,  0.80]]) # desemp_t = -0.15 PIB_{t-1} + 0.8 desemp_{t-1}
Sigma = np.array([[1.0, -0.4],
                  [-0.4, 0.5]])
L = np.linalg.cholesky(Sigma)

Y_var = np.zeros((T, 2))
for t in range(1, T):
    u = L @ np.random.normal(0, 1, 2)
    Y_var[t] = A @ Y_var[t-1] + u

df_var = pd.DataFrame(Y_var, columns=['gdp_growth', 'unemployment'])
dates  = pd.date_range('1980-01-01', periods=T, freq='QE')
df_var.index = dates

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df_var.index, df_var['gdp_growth'], color='#4361ee', linewidth=1.5)
axes[0].set_ylabel('Crecimiento PIB (%)')
axes[0].set_title('Sistema VAR(1): Crecimiento PIB y Desempleo')
axes[1].plot(df_var.index, df_var['unemployment'], color='#f72585', linewidth=1.5)
axes[1].set_ylabel('Desempleo (%)')
plt.tight_layout()
plt.show()

## 1 — ARIMA: identificación por ACF/PACF

In [ ]:
series = df_var['gdp_growth']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(series.values, color='#4361ee', linewidth=1.5)
axes[0].set_title('Serie: PIB growth')

plot_acf(series,  lags=20, ax=axes[1], color='#4361ee', title='ACF — PIB growth')
plot_pacf(series, lags=20, ax=axes[2], color='#f72585', title='PACF — PIB growth')

plt.tight_layout()
plt.show()

# ADF test
adf_stat, adf_p, *_ = adfuller(series, regression='c')
print(f'ADF test: stat={adf_stat:.4f}  p={adf_p:.4f}  → {"Estacionaria" if adf_p < 0.05 else "Raíz unitaria"}')
print()
print('Identificación de orden ARIMA (Box & Jenkins, 1970):')
print('  ACF decae lentamente → AR (o I si es muy lento)')
print('  PACF corte en lag p → AR(p)')
print('  ACF corte en lag q → MA(q)')

# Ajustar ARIMA(1,0,1)
mod_arima = ARIMA(series, order=(1, 0, 1)).fit()
print(f'\nARIMA(1,0,1):')
print(f'  AR(1): {mod_arima.params["ar.L1"]:.4f}  MA(1): {mod_arima.params["ma.L1"]:.4f}')
print(f'  AIC: {mod_arima.aic:.2f}  BIC: {mod_arima.bic:.2f}')

## 2 — VAR: selección de lag y estimación

In [ ]:
# Selección de lag óptimo por AIC/BIC
var_model_sel = VAR(df_var)
lag_sel = var_model_sel.select_order(maxlags=6)
print('Selección de orden de lag VAR:')
print(lag_sel.summary())

# Estimar VAR(1)
mod_var = var_model_sel.fit(maxlags=2, ic='aic')
print(mod_var.summary())

## 3 — Funciones de Impulso-Respuesta (IRF)

In [ ]:
irf = mod_var.irf(periods=12)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

variables = df_var.columns.tolist()
titles = [
    'Shock a PIB → PIB',        'Shock a PIB → Desempleo',
    'Shock a Desempleo → PIB',  'Shock a Desempleo → Desempleo',
]

for idx, (impulse, response) in enumerate([
    (0,0), (0,1), (1,0), (1,1)
]):
    ax = axes[idx // 2, idx % 2]
    irfs = irf.irfs[:, response, impulse]
    lo   = irf.stderr_cov()[:, response, impulse] if hasattr(irf, 'stderr_cov') else np.zeros(13)

    ax.plot(range(13), irfs, color='#4361ee', linewidth=2.5)
    ax.axhline(0, color='black', linewidth=1)
    ax.fill_between(range(13), irfs - 1.96*lo, irfs + 1.96*lo,
                    alpha=0.15, color='#4361ee')
    ax.set_title(titles[idx])
    ax.set_xlabel('Horizonte (trimestres)')

plt.suptitle('Funciones de Impulso-Respuesta — Sims (1980)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 — Causalidad de Granger

In [ ]:
print('Test de Causalidad de Granger — Granger (1969)')
print('=' * 55)

for col1, col2 in [('unemployment', 'gdp_growth'), ('gdp_growth', 'unemployment')]:
    data_gc = df_var[[col1, col2]].values   # col1 → causa potencial de col2
    print(f'\nH₀: {col1} NO Granger-causa {col2}')
    results = grangercausalitytests(data_gc, maxlag=4, verbose=False)
    for lag, res in results.items():
        fstat, pval = res[0]['ssr_ftest'][:2]
        print(f'  lag={lag}  F={fstat:.3f}  p={pval:.4f}  '
              f'{"→ Granger-causa" if pval < 0.05 else "→ No Granger-causa"}')

print()
print('IMPORTANTE: Causalidad de Granger = predictibilidad, NO causalidad en sentido Pearl.')
print('Un variable omitida puede hacer que X "Granger-cause" Y sin relación causal real.')

## Resumen

| Herramienta | Qué modela | Uso |
|---|---|---|
| ARIMA(p,d,q) | Una serie univariada | Pronóstico, descripción |
| VAR(p) | Sistema multivariado | IRF, FEVD, Granger |
| Granger causality | Precedencia temporal | Predictibilidad incremental |
| VECM | VAR + cointegración | Series I(1) cointegradas |

**Referencias:**
- Granger, C.W.J. (1969). Investigating causal relations by econometric models. *Econometrica*, 37(3).
- Sims, C.A. (1980). Macroeconomics and reality. *Econometrica*, 48(1).
- Box, G.E.P. & Jenkins, G.M. (1970). *Time Series Analysis: Forecasting and Control*. Holden-Day.
- Hamilton, J.D. (1994). *Time Series Analysis*. Princeton UP.
- Lütkepohl, H. (2005). *New Introduction to Multiple Time Series Analysis*. Springer.

**Siguiente:** `09_heterogeneous_treatment_ml.ipynb`